# Sleep staging at stage transitions

Can two existing classifiers correct each other's mistakes, and where do errors persist? This exploratory analysis reuses the saved development predictions from [the refinement campaign](../docs/refinement-results.md). It compares the two tuned models with a fixed arithmetic mean of their five class probabilities. A single equal-weight blend was evaluated. The primary metric is mean macro F1 across the five frozen subject folds. N1 F1 and transition diagnostics pool epochs across folds.

Requires the campaign summaries and prediction files under `var/research/sc-w30-refinement-v1/` and the repository's Python environment. All inputs are local and the analysis requires no model fitting.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score

from sleep_staging.config import CLASSES
from sleep_staging.provenance import hash_file
from sleep_staging.research import paired_subject_bootstrap

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/research/refinement.yaml").is_file())
campaign = root / "var/research/sc-w30-refinement-v1"
labels = CLASSES["task_5"]
probabilities = [f"prob_{label}" for label in labels]
split = pd.read_csv(root / "configs/splits/sc_subjects_v1.csv")
dev_fold = split.loc[split.partition.eq("development")].set_index("subject_id").fold

def predictions(family):
    summary = json.loads((campaign / f"{family}.json").read_text())
    path = campaign / "runs" / summary["identity"] / family / "oof.parquet"
    assert summary["test_evaluated"] is False and hash_file(path) == summary["oof_sha256"]
    frame = pd.read_parquet(path).set_index("epoch_id", verify_integrity=True).sort_index()
    assert set(frame.subject_id) == set(dev_fold.index)
    assert frame.fold.eq(frame.subject_id.map(dev_fold)).all()
    scores = frame[probabilities].to_numpy()
    np.testing.assert_allclose(scores.sum(axis=1), 1, atol=1e-5)
    np.testing.assert_array_equal(np.asarray(labels)[scores.argmax(axis=1)], frame.y_pred)
    return frame

tree, recurrent = predictions("lgbm"), predictions("bilstm")
metadata = ["subject_id", "recording_id", "onset_s", "fold", "y_true"]
pd.testing.assert_frame_equal(tree[metadata], recurrent[metadata])
epochs = tree[metadata].copy()
epochs["LightGBM"] = tree.y_pred
epochs["BiLSTM"] = recurrent.y_pred
mean_probability = (tree[probabilities] + recurrent[probabilities]) / 2
epochs["50/50 blend"] = mean_probability.idxmax(axis=1).str.removeprefix("prob_")
print(f"{len(epochs):,} epochs from {epochs.recording_id.nunique()} nights and {epochs.subject_id.nunique()} development subjects.")

.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


152,052 epochs from 122 nights and 62 development subjects.


## Five-fold results

Each fold has equal weight in the macro F1 mean. The standard deviation uses the same convention as the refinement report (`ddof=0`). The paired bootstrap resamples subjects with all their nights and calculates the difference in pooled macro F1. Its interval is conditional on the fitted models.

In [2]:
models = ["LightGBM", "BiLSTM", "50/50 blend"]
folds = pd.DataFrame({
    name: [f1_score(g.y_true, g[name], labels=labels, average="macro")
           for _, g in epochs.groupby("fold", sort=True)]
    for name in models
}, index=sorted(epochs.fold.unique())).rename_axis("fold")
display(folds.round(4))
summary = pd.DataFrame({"Mean macro F1": folds.mean(), "Fold SD": folds.std(ddof=0)})
summary["Pooled N1 F1"] = [f1_score(epochs.y_true, epochs[m], labels=["N1"], average=None)[0]
                           for m in summary.index]
display(summary.sort_values("Mean macro F1", ascending=False).round(4))
keys = ["subject_id", "y_true"]
reference = epochs[keys].assign(y_pred=epochs["LightGBM"]).reset_index()
candidate = epochs[keys].assign(y_pred=epochs["50/50 blend"]).reset_index()
paired = paired_subject_bootstrap(reference, candidate, repeats=2000)
low, high = paired["ci95"]
print(f'Pooled macro F1 difference, blend minus LightGBM = {paired["delta"]:+.4f} (95% CI {low:+.4f} to {high:+.4f}).')

,LightGBM,BiLSTM,50/50 blend
fold,,,
0,0.7546,0.7477,0.7625
1,0.7976,0.7956,0.8071
2,0.7992,0.7883,0.8044
3,0.8042,0.7827,0.7996
4,0.7906,0.7857,0.7942


,Mean macro F1,Fold SD,Pooled N1 F1
50/50 blend,0.7935,0.0161,0.5293
LightGBM,0.7893,0.0179,0.5159
BiLSTM,0.7800,0.0167,0.5126


Pooled macro F1 difference, blend minus LightGBM = +0.0045 (95% CI +0.0005 to +0.0083).


## Stage changes

An epoch is adjacent to a transition if either immediately neighboring expert label differs from its own. Both neighbors must be present, exactly 30 seconds away, within the same recording. This excludes edges and gaps. The labels define this evaluation subgroup after prediction. Subgroup accuracy is sensitive to its stage mix.

In [3]:
ordered = epochs.sort_values(["recording_id", "onset_s"])
prev_contiguous = ordered.recording_id.eq(ordered.recording_id.shift()) & ordered.onset_s.diff().eq(30)
eligible = prev_contiguous & prev_contiguous.shift(-1, fill_value=False)
adjacent_change = ordered.y_true.ne(ordered.y_true.shift()) | ordered.y_true.ne(ordered.y_true.shift(-1))
rows = []
for name, selected in [("Adjacent to change", eligible & adjacent_change), ("Stable", eligible & ~adjacent_change)]:
    group = ordered.loc[selected]
    row = {"Epochs": len(group)}
    for model in models:
        row[model] = group[model].eq(group.y_true).mean()
    rows.append({"Group": name, **row})
display(pd.DataFrame(rows).set_index("Group").round(4))
print(f"Excluded from this subgroup analysis: {(~eligible).sum():,} epochs.")
display(pd.crosstab(epochs.y_true, epochs.LightGBM).reindex(index=labels, columns=labels))

,Epochs,LightGBM,BiLSTM,50/50 blend
Group,,,,
Adjacent to change,26212,0.606,0.5813,0.6024
Stable,125343,0.878,0.8698,0.8827


Excluded from this subgroup analysis: 497 epochs.


LightGBM,W,N1,N2,N3,REM
y_true,,,,,
W,44332,3814,302,89,240
N1,2186,9286,3847,105,1179
N2,359,4754,45524,2736,1582
N3,29,32,1345,9245,5
REM,269,1511,1305,40,17936


## Interpretation and next experiment

The equal-weight blend led on mean development macro F1 (0.7935) and pooled N1 F1 (0.5293). LightGBM retained the highest accuracy on epochs adjacent to stage changes (0.6060). On stable epochs the blend reached 0.8827. These comparisons use the existing annotation-based wake crop and retrospective features.

The next comparison should test whether learned waveform features help at those transitions. Train a compact temporal convolutional network on the existing epoch descriptors, then compare against an EEG/EOG waveform encoder and a matched-channel LightGBM. Compare the branches separately and in an equal-weight blend. Preserve within-subject pairing, measure N1 precision and recall, and repeat the strongest comparison on recordings prepared without an annotation-defined crop. The 16 reserved Cassette subjects can provide a final evaluation after the protocol is frozen.